In [69]:
import pandas as pd
import glob
import re
import os
import shutil

In [70]:
files = []
folder_to_search = "krona/*"

for file in glob.glob(folder_to_search):
    files.append(file.split("\\")[-1].split("/")[-1])
# files
print(files)

['CB-CL100066028-L01-509.html', 'CB-CL100066028-L01-510.html', 'CB-CL100066028-L01-511.html', 'CB-CL100066028-L01-512.html', 'CB-CL100066028-L01-513.html', 'CB-CL100066028-L01-514.html', 'CB-CL100066028-L01-515.html', 'CB-CL100066028-L01-516.html', 'CB-CL100066028-L01-531.html', 'K1-CL100055796-L01-501.html', 'K1-CL100055796-L01-502.html', 'K1-CL100055796-L01-503.html', 'K1-CL100055796-L01-504.html', 'K1-CL100055796-L01-505.html', 'K1-CL100055796-L01-506.html', 'K1-CL100055796-L01-507.html', 'K1-CL100055796-L01-508.html', 'K10-CL100060011-L02-525.html', 'K10-CL100060011-L02-526.html', 'K10-CL100060011-L02-527.html', 'K10-CL100060011-L02-528.html', 'K10-CL100060011-L02-529.html', 'K10-CL100060011-L02-530.html', 'K10-CL100060011-L02-531.html', 'K10-CL100060011-L02-532.html', 'K11new-CL100060010-L01-509.html', 'K11new-CL100060010-L01-510.html', 'K11new-CL100060010-L01-511.html', 'K11new-CL100060010-L01-512.html', 'K11new-CL100060010-L01-513.html', 'K11new-CL100060010-L01-514.html', 'K11ne

In [71]:
df = pd.read_csv("Connecting_Paths.csv", sep=";")
df = df[df["wgt01@6@15"].notna()]
df = df[df.columns.drop(list(df.filter(regex='Unnamed:')))]
df["Reformatted_Index_name"] = df.Reformatted_Index_name.astype(int)
df

,wgt01@6@15,Folder/Lane_name,Index_name,Reformatted_Index_name,Library_name,Sample_name,Carboy/Tank_ID,Vineyard,Winery,Status/Timepoint,SampleDate,Replicate
0,243.0,K31,BGI_19,519,07B6,01a@press1@13-10,1,1.0,1,1_first-press,13-Oct,a
1,140.0,K18,BGI_28,528,13B9,01b@press1@13-10,1,1.0,1,2_mid-press,13-Oct,a
2,162.0,K21,BGI_26,526,13B7,01c@press1@13-10,1,1.0,1,3_final-press,13-Oct,a
3,117.0,K15,BGI_5,505,28B7,01a@sed@14-10,1,1.0,1,sedimentation,14-Oct,a
4,171.0,K22,BGI_3,503,25B9,01a@on@19-10,1,1.0,1,onset,19-Oct,a
...,...,...,...,...,...,...,...,...,...,...,...,...
289,290.0,NB,BGI_26,526,32B2neg,B2@library@neg2,NaN,NaN,NaN,NaN,NaN,NaN
290,291.0,NB,BGI_27,527,14B5ext,B5@extration@neg,NaN,NaN,NaN,NaN,NaN,NaN
291,292.0,NB,BGI_28,528,16B1ext,B1@extration@neg,NaN,NaN,NaN,NaN,NaN,NaN
292,293.0,NB,BGI_29,529,33B8negEBT,B8@library@neg@wEBT,NaN,NaN,NaN,NaN,NaN,NaN


In [72]:
df2 = df
# Function to create regex pattern, search through the list of strings, and construct the final string
def create_new_location(row, files, folder_to_search):
    """Create old and new location of files and add to dataframe to move files.

    Args:
        row (row): Row in pandas dataframe, used with 'apply' function
        files (list): List of all files to match
        folder_to_search (str): Prefix for original file location

    Returns:
        pandas series: Series object with [0][0] containing old location
                                      and [0][1] containing new location.
    """    
    # Sometimes our folder search still contains the * used to search so we rmv
    folder_to_search = folder_to_search.replace("*", "")
    # Get the row number, easiest method for counting upis to reset pandas index
    timepoint = row.name + 1
    
    # We get the important values from each row
    col1_value = row['Folder/Lane_name']
    col2_value = row['Reformatted_Index_name']
    vineyard = row['Vineyard']
    winery = row['Winery']
    replicate = row['Replicate']
    sample_name = row['Sample_name']


    # print(col1_value, col2_value, vineyard, winery, replicate, sample_name)
    # Vineyard is float, have to cast to int for clean number but pass NaNs 
    try:
        vineyard = int(vineyard)
    except ValueError:
        pass

    # Regex pattern matching lane and index name
    regex_pattern = f".*{col1_value}(-|_).*{col2_value}.*"
    matches = [s for s in files if re.search(regex_pattern, s)]
    
    # Construct the final strings
    original = [f"{folder_to_search}{match}" for match in matches]
    # final_strings = [f"{folder_to_search}V{vineyard}_W{winery}_R{replicate}/{timepoint}_{match}" for match in matches]
    
    final_strings = [
    "{folder_to_search}V{vineyard}_W{winery}_R{replicate}/{timepoint}_{sample_name}.{extension}".format(
        folder_to_search=folder_to_search,
        vineyard=vineyard,
        winery=winery,
        replicate=replicate,
        timepoint=timepoint,
        sample_name=sample_name,
        extension=match.split(".")[-1]
    )
    for match in matches
    ]

    return original, final_strings 


In [73]:
df2 = df
# df2['original'] = pd.Series(dtype=str)
# df2['new'] = pd.Series(dtype=str)
# df2
tmp = df2

original_list = []
new_list = []

# Okay this loop is pretty complicated but we loop for each group in the winery
# and then we group each replicate together and reset the index

for group in tmp.Winery.unique():
    group = tmp.loc[tmp['Winery'] == group]
    for rep in group.Replicate.unique():
        replicate = group.loc[group["Replicate"] == rep].reset_index()

        # Then we get our series objects by calling the function
        old_new = replicate.apply(create_new_location, axis=1, args=(files, folder_to_search))

        # Cast the series object to a list and merge them
        original_list = original_list + (old_new.apply(lambda x: x[0][0]).to_list())
        new_list = new_list + (old_new.apply(lambda x: x[1][0]).to_list())

# The earlier loop does not catch the NB column so we fix that here
old_new = tmp.loc[tmp['Folder/Lane_name'] == "NB"].reset_index().apply(create_new_location, axis=1, args=(files, folder_to_search))
original_list = original_list + (old_new.apply(lambda x: x[0][0]).to_list())
new_list = new_list + (old_new.apply(lambda x: x[1][0]).to_list())

In [74]:
# Here we add everything to the original dataframe
filenames = [e[len(folder_to_search)-1:].split(".")[0] for e in original_list]
df2["filenames"] = filenames
df2["original"] = original_list
df2["new"] = new_list


In [75]:
df2.head(2)

,wgt01@6@15,Folder/Lane_name,Index_name,Reformatted_Index_name,Library_name,Sample_name,Carboy/Tank_ID,Vineyard,Winery,Status/Timepoint,SampleDate,Replicate,filenames,original,new
0,243.0,K31,BGI_19,519,07B6,01a@press1@13-10,1,1.0,1,1_first-press,13-Oct,a,K31-CL100060270-L02-519,krona/K31-CL100060270-L02-519.html,krona/V1_W1_Ra/1_01a@press1@13-10.html
1,140.0,K18,BGI_28,528,13B9,01b@press1@13-10,1,1.0,1,2_mid-press,13-Oct,a,K18-CL100060399-L02-528,krona/K18-CL100060399-L02-528.html,krona/V1_W1_Ra/2_01b@press1@13-10.html


In [76]:
folderlist = ["/".join(e.split("/", 2)[:2]) for e in new_list]
folderlist = list(set(folderlist))
for folder in folderlist:
    if not os.path.exists(folder):
        os.makedirs(folder)



In [ ]:
def move_files(row):

    old_location = row["original"]
    new_location = row["new"]
    copy_location = old_location.split("/")[0]+"/"+new_location.split("/")[-1]

# "C:\Users\domin\Documents\WUR\Thesis\krona\K31-CL100060270-L02-521.html"
    os.getcwd()
    old_location = os.path.normpath(f"{os.getcwd()}/{old_location}")
    new_location = os.path.normpath(f"{os.getcwd()}/{new_location}")
    
    try:
        shutil.copy(old_location, (copy_location))
    except FileNotFoundError:
        print(f"Source file {old_location} not found.")
        print(old_location, new_location, copy_location)
    except PermissionError:
        print(f"Permission denied when copying {old_location} to {new_location}.")
    except Exception as e:
        print(f"An error occurred: {e}")


    try:
        shutil.move(old_location, new_location)
    except FileNotFoundError:
        print(f"Source file {old_location} not found.")
        print(old_location, new_location, copy_location)
    except PermissionError:
        print(f"Permission denied when moving {old_location} to {new_location}.")
    except Exception as e:
        print(f"An error occurred: {e}")

df2.apply(move_files, axis=1)
# df2
# c:\Users\domin\Documents\WUR\Thesis\krona\K7-CL100052128-L01-517.html
# BUG DISCOVERED! That one is not supposed to be that
# It's supposed to be K7-CL100052128-L01-521.html!!!


Source file c:\Users\domin\Documents\WUR\Thesis\krona\K7-CL100052128-L01-517.html not found.
Source file c:\Users\domin\Documents\WUR\Thesis\krona\K7-CL100052128-L01-517.html not found.


0      None
1      None
2      None
3      None
4      None
       ... 
289    None
290    None
291    None
292    None
293    None
Length: 294, dtype: object

In [ ]:
# df2.to_csv("Final_Files.csv")


: 